In [1]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

chat_message_history = SQLChatMessageHistory(
    session_id="test_session_id",
    connection_string="sqlite:///sqlite.db",
)

chat_message_history.add_user_message("Hello")
chat_message_history.add_ai_message("Hi")

/Users/samsepiol/Documents/LanguageBox/.venv/lib/python3.12/site-packages/langchain_community/chat_message_histories/sql.py:140: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. Use Use connection instead instead.
  warn_deprecated(


In [2]:
chat_message_history.messages

[HumanMessage(content='Hello'),
 AIMessage(content='Hi'),
 HumanMessage(content='Hello'),
 AIMessage(content='Hi')]

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

In [4]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(
            variable_name="history",
        ),
        ("human", "{question}"),
    ]
)

chain = prompt | ChatOpenAI()

In [5]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///sqlite.db",
    ),
    input_messages_key="question",
    history_messages_key="history",
)

In [6]:
# This is where we configure the session id
config = {
    "configurable": {
        "session_id": "<SQL_SESSION_ID>",
    }
}

In [7]:
chain_with_history.invoke(
    {"question": "Hi! I'm bob"},
    config=config,
)

AIMessage(content='Hello Bob! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 70, 'total_tokens': 80}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-367201b2-11c6-4f7e-a8b8-d28a6c3e906e-0', usage_metadata={'input_tokens': 70, 'output_tokens': 10, 'total_tokens': 80})

In [8]:
chain_with_history.invoke(
    {"question": "Whats my name"},
    config=config,
)

AIMessage(content='Your name is Bob. How can I assist you further, Bob?', response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 91, 'total_tokens': 105}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-9c2f8db9-67f9-4a37-87a5-3de025561e2c-0', usage_metadata={'input_tokens': 91, 'output_tokens': 14, 'total_tokens': 105})